# GazeAlign — Step-by-Step Walkthrough

This notebook walks through the full GazeAlign pipeline on a single
example: load an image, load its fixation scanpath, run the trained
model, and visualize the learned gaze-conditioned attention mask
alongside the raw gaze prior.

It mirrors `scripts/predict_single.py`, but breaks each stage into its
own cell so you can inspect intermediate tensors.


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt

from gazealign import (
    DINOv3Encoder, ScanpathTransformer, ViTMaskGenerator, GazePriorHeatmap,
    classifier, build_transform, get_scanpath, load_fixation_csv, get_device,
)
from gazealign.visualize import patch_to_image, make_overlay

device = get_device()
print("Using device:", device)


## 2. Load a trained checkpoint

In [ ]:
CHECKPOINT = "../checkpoints/best_model.pth"
CLASSES = ["CHF", "Normal", "pneumonia"]   # update to match your trained config
IMG_SIZE = 518
GRID_SIZE = 37

image_encoder = DINOv3Encoder().to(device).eval()
scanpath_encoder = ScanpathTransformer(d_model=512).to(device).eval()
mask_generator = ViTMaskGenerator(emb_dim=512, num_patches=GRID_SIZE**2).to(device).eval()
clf = classifier(num_classes=len(CLASSES)).to(device).eval()
gaze_prior_model = GazePriorHeatmap(img_size=IMG_SIZE).to(device).eval()

ckpt = torch.load(CHECKPOINT, map_location=device)
image_encoder.load_state_dict(ckpt["image_encoder_state"])
scanpath_encoder.load_state_dict(ckpt["scanpath_encoder_state"])
mask_generator.load_state_dict(ckpt["mask_generator_state"])
clf.load_state_dict(ckpt["classifier_state"])
print(f"Loaded checkpoint from epoch {ckpt.get('epoch', '?')}")


## 3. Load one image and its scanpath

In [ ]:
import cv2

IMAGE_PATH = "../examples/images/sample_cxr_001.jpg"
FIXATIONS_PATH = "../examples/fixations/cxr_samples.csv"
DICOM_ID = "sample_cxr_001"

img_rgb = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
orig_h, orig_w = img_rgb.shape[:2]

fix_df = load_fixation_csv(FIXATIONS_PATH)
scanpath = get_scanpath(fix_df, DICOM_ID, img_height=orig_h, img_width=orig_w)
scanpath = scanpath[:200].to(device)
print("Scanpath shape:", scanpath.shape)

plt.figure(figsize=(5,5))
plt.imshow(img_rgb)
plt.title("Input image")
plt.axis("off")
plt.show()


## 4. Encode the image and the scanpath

In [ ]:
transform = build_transform(IMG_SIZE)
img_tensor = transform(img_rgb).unsqueeze(0).to(device)

with torch.no_grad():
    img_emb, patch_tokens, cls_token = image_encoder(img_tensor)
    _, sp_emb, _ = scanpath_encoder([scanpath])

print("img_emb:", img_emb.shape)
print("patch_tokens:", patch_tokens.shape)
print("sp_emb:", sp_emb.shape)


## 5. Generate the gaze-conditioned attention mask

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    patch_mask = torch.sigmoid(mask_generator(sp_emb))  # [1, 37, 37]

attention_mask_full = patch_to_image(patch_mask[0].cpu().numpy(), IMG_SIZE, IMG_SIZE)

plt.figure(figsize=(5,5))
plt.imshow(attention_mask_full, cmap="hot")
plt.title("Learned gaze-conditioned attention mask")
plt.axis("off")
plt.show()


## 6. Classify using the gaze-masked features

In [ ]:
with torch.no_grad():
    B, N, D = patch_tokens.shape
    weights = patch_mask.view(B, N, 1)
    feat_attended = (patch_tokens * weights).mean(dim=1)
    logits = clf(feat_attended)
    probs = F.softmax(logits, dim=1)[0].cpu().numpy()

for c, p in zip(CLASSES, probs):
    print(f"{c:>12s}: {p:.4f}")

predicted = CLASSES[int(probs.argmax())]
print("\nPredicted class:", predicted)


## 7. Compare against the raw gaze prior

In [ ]:
with torch.no_grad():
    scanpath_padded = F.pad(scanpath, (0, 0, 0, max(0, 200 - scanpath.size(0)))).unsqueeze(0)
    gaze_prior_map = gaze_prior_model(scanpath_padded)[0].cpu().numpy()

gaze_prior_full = patch_to_image(gaze_prior_map, IMG_SIZE, IMG_SIZE)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))); axs[0].set_title("Image"); axs[0].axis("off")
axs[1].imshow(attention_mask_full, cmap="hot"); axs[1].set_title("Learned attention (model)"); axs[1].axis("off")
axs[2].imshow(gaze_prior_full, cmap="jet"); axs[2].set_title("Raw gaze prior (fixations only)"); axs[2].axis("off")
plt.tight_layout()
plt.show()


## 8. Overlay on the original image

This is the same overlay produced by `scripts/predict_single.py --save_overlay`.


In [ ]:
overlay = make_overlay(cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE)), attention_mask_full)

plt.figure(figsize=(5,5))
plt.imshow(overlay)
plt.title(f"Predicted: {predicted}")
plt.axis("off")
plt.show()
